In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/pipelines/02_extracting_from_bronze_to_treatments_into_silver/step00_IMPORTS

In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/pipelines/02_extracting_from_bronze_to_treatments_into_silver/step00_AUXILIARY_FUNCTIONS

In [0]:
spark.table('bronze.usinas').show(truncate=False)

In [0]:
df = (
        spark.table('bronze.usinas')
             .withColumnsRenamed(
                                 {
                                    'id_usina' : 'plant_id',
                                    'nome_usina' : 'plant_name',
                                    'distribuidora' : 'distributor',
                                    'potencia_kwp' : 'power_kwp',
                                    'data_protocolo_acesso' : 'date_access_protocol',
                                    'data_conexao' : 'date_connection'
                                 }
                                )
             .withColumn('power_kwp', F.col('power_kwp').cast('double'))
     )

# Incremental load: never replaces silver.plants — appends only new plants
target = "silver.plants"
key_cols = ['plant_id']

if not spark.catalog.tableExists(target):
    # first load: table does not exist yet — create it
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
    print(f"\n✓ {target} created ({df.count()} rows)\n")
else:
    # append only records whose key is not present in silver yet
    df_new = df.join(spark.table(target).select(*key_cols).distinct(), on=key_cols, how='left_anti')
    n_new = df_new.count()
    if n_new:
        df_new.write.format("delta").mode("append").saveAsTable(target)
        print(f"\n✓ {target} appended {n_new} new row(s)\n")
    else:
        print(f"\n↷ {target}: no new records — nothing appended\n")